## Задача 1.
Напишите функцию для решения следующей задачи:

В офисе AtCoder есть один увлажнитель. В момент времени 0 в нем нет воды.
Вы доливаете воду N раз. В i-й раз (1 ≤ i ≤ N) вы доливаете V_i литров в момент T_i.
Гарантируется, что T_i < T_{i+1} для всех 1 ≤ i ≤ N−1.
Увлажнитель протекает: пока в нем есть вода, её количество уменьшается со скоростью 1 литр за единицу времени.
Требуется найти количество воды в увлажнителе сразу после того, как вы закончите долив в момент T_N.

Данные подаются как многострочная строка (input_str):

Первая строка: одно целое число N — количество доливов.
Следующие N строк: по два целых числа T_i и V_i, разделённые пробелом.
Гарантируется, что T_i строго возрастают.
Функция должна вернуть строку с одним числом — количеством воды сразу после долива в момент T_N.

Ваш код должен пройти следующие тесты:

Тест 1
Вход:
4
1 3
3 1
4 4
7 1
Выход:
3

Тест 2
Вход:
3
1 8
10 11
21 5
Выход:
5

Тест 3
Вход:
10
2 1
22 10
26 17
29 2
45 20
47 32
72 12
75 1
81 31
97 7
Выход:
57

### Ключевые моменты реализации

**Парсинг входных данных:** Нам нужно корректно разбить многострочную строку на числа. Используем методы split() и int().

**Логика испарения:** Самая важная часть. Количество воды не может стать отрицательным. Формула обновления воды между событиями:

Wnew​ =max(0,Wold​ −(Tcurrent​ − Tprev​))

Здесь функция max гарантирует, что мы не уйдем в минус, если вода полностью испарилась до следующего долива.

**Цикл:** Мы проходим по всем событиям, обновляя время и объем воды.


Вот реализация функции на Python:

In [1]:
def solve_humidifier(input_str: str) -> str:
    # 1. Парсинг входных данных
    lines = input_str.strip().split('\n')
    
    # Первая строка - количество доливов N
    n = int(lines[0])
    
    # Переменные состояния
    current_water = 0
    prev_time = 0
    
    # Проходим по следующим N строкам
    for i in range(1, n + 1):
        # Разбираем строку формата "T_i V_i"
        parts = lines[i].split()
        t_current = int(parts[0])
        v_add = int(parts[1])
        
        # 2. Вычисляем прошедшее время
        time_diff = t_current - prev_time
        
        # 3. Учитываем испарение (вода не может быть < 0)
        # Это критический момент: если time_diff > current_water, вода закончится раньше
        current_water = max(0, current_water - time_diff)
        
        # 4. Доливаем новую воду
        current_water += v_add
        
        # Обновляем время последнего события
        prev_time = t_current
    
    # Возвращаем результат как строку (согласно требованию задачи)
    return str(current_water)

In [2]:
# --- Проверка на предоставленных тестах ---

test_1_input = """4
1 3
3 1
4 4
7 1"""

test_2_input = """3
1 8
10 11
21 5"""

test_3_input = """10
2 1
22 10
26 17
29 2
45 20
47 32
72 12
75 1
81 31
97 7"""

print(f"Тест 1: {solve_humidifier(test_1_input)} (Ожидается: 3)")
print(f"Тест 2: {solve_humidifier(test_2_input)} (Ожидается: 5)")
print(f"Тест 3: {solve_humidifier(test_3_input)} (Ожидается: 57)")

Тест 1: 3 (Ожидается: 3)
Тест 2: 5 (Ожидается: 5)
Тест 3: 57 (Ожидается: 57)


## Задача 2.
Даны два строковых представления чисел A и B. Нужно максимизировать A, заменив в нём любую
цифру на цифру из B. Каждую цифру B можно использовать только один раз.

### План алгоритма
Исходя из вышесказанного, попробуй сформулировать шаги алгоритма. Вот каркас, который ты можешь дополнить:
Преобразовать строку B в список цифр и отсортировать его по убыванию. Зачем? (Чтобы быстро брать наилучшую доступную цифру).
Преобразовать строку A в список (так как строки в Python неизменяемы).
Пройтись циклом по цифрам A (слева направо).
Внутри цикла сравнить текущую цифру A с наибольшей оставшейся цифрой из B.
Условие замены: Если цифра из B строго больше цифры в A, выполняем замену и убираем использованную цифру из пула B. Если нет — что делаем? (Продолжаем поиск или останавливаемся? Подумай, может ли быть выгодно заменить младший разряд, если старший заменить не удалось?).
Собрать список обратно в строку.

**Ключевая идея**
Цифры из B — это пул ресурсов, который мы тратим по мере необходимости. Если текущая позиция в A не выгодна для замены, мы не останавливаемся, а просто переходим к следующей позиции в A, сохраняя текущую лучшую цифру из B для будущих сравнений.
Нужны два независимых индекса.

In [5]:
def maximize_a(str_a: str, str_b: str) -> str:
    # Преобразуем строки в списки символов для удобства
    list_a = list(str_a)
    list_b = list(str_b)
    
    # Сортируем обе строки в порядке убывания
    # list_a.sort(reverse=True)
    list_b.sort(reverse=True)
    b_index = 0  # Индекс для списка B
    # Идем по обоим спискам и сравниваем символы
    for i in range(len(list_a)):
        if b_index >= len(list_b):
            break  # Если мы уже использовали все символы из B, выходим
        
        if list_a[i] < list_b[b_index]:
            # Если символ из A меньше, чем символ из B, меняем их местами
            list_a[i] = list_b[b_index]
            b_index += 1
    
    # Преобразуем список обратно в строку и возвращаем
    return ''.join(list_a)

In [7]:
str_a = "456380"
str_b = "3"

maximize_a(str_a, str_b)

'456383'

In [8]:
maximize_a("998", "24576")

'998'